# multi_heatmap · 02 — Construcción, entrenamiento y validación

Notebook de **Google Colab** para la variante `multi_heatmap`: arquitectura multi-configuración (19/64/128/256 electrodos) con campo de superficie entrenado, bucle de entrenamiento balanceado y evaluación frente a la línea base analítica.

**Uso:** Runtime → **GPU (T4)**. Ejecutar primero `01_exploracion_datos.ipynb` para poblar la caché del dataset.

## 0 · Entorno

In [ ]:
# ---- 0 · Entorno (Colab o local) --------------------------------------
# Requiere el notebook 01 ejecutado (dataset cacheado en Drive o local).
# Colab: Runtime > Change runtime type > GPU (T4).
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/sonoAESS/universal-eeg-transformer.git"
BRANCH   = "explore/multi-heatmap"

IN_COLAB = "google.colab" in sys.modules or "/content" in os.getcwd()

if IN_COLAB:
    ROOT = Path("/content/universal-eeg-transformer")
    if not ROOT.exists():
        !git clone -b {BRANCH} {REPO_URL} {ROOT}
    # Dependencias: TF ya viene preinstalado; el resto lo trae pyproject vía pip.
    %pip install -q mne pyyaml pandas matplotlib scikit-learn
else:
    ROOT = Path.cwd()
    while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print(f"ROOT = {ROOT}\nColab = {IN_COLAB}")

In [ ]:
# ---- 0b · Persistencia en Google Drive (opcional) ---------------------
# Cachea dataset + checkpoints en Drive para no re-descargar en cada sesión.
USE_DRIVE = IN_COLAB

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/universal_eeg_cache")
    CACHE.mkdir(exist_ok=True)
    for link in ("data/processed", "runs"):
        target = CACHE / link.split("/")[-1]
        target.mkdir(exist_ok=True)
        dest = ROOT / link
        if not dest.exists():
            dest.symlink_to(target)
    print("Cache y runs enlazados a:", CACHE)
else:
    print("Modo local: cache en ./data/processed y ./runs")

## 1 · Carga

In [ ]:
# ---- 1 · Experimento ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)
plt.rcParams["figure.dpi"] = 110

CONFIG = ROOT / "config/multi_heatmap.yaml"
FORCE  = False   # True = reentrenar desde cero ignorando el checkpoint

from eeg_transform.nb import (
    evaluate_multiconfig, evaluate_multiconfig_surface,
    load_experiment, multiconfig_data, plot_multiconfig_bars,
    plot_multiconfig_heatmap, plot_multiconfig_scalps,
    plot_multiconfig_surface, plot_training, train_variant,
)
from eeg_transform.training.trainer import build_multiconfig_model

cfg, ds = load_experiment(CONFIG)
data = multiconfig_data(cfg, ds)
print(ds.summary())

## 2 · Arquitectura

In [ ]:
# ---- 2 · Arquitectura --------------------------------------------------
# Núcleo canónico (64 ch, latente = C) compartido + proyecciones fijas
# P_s/Q_s por configuración. Salida extra: campo de superficie en la malla
# compartida vía S_s (surface_loss_weight), predicción intra-configuración.
model = build_multiconfig_model(cfg, data)

n_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"Variante: {cfg.model.variant} | latente: {model.n_canonical} | "
      f"configs: {len(model.configs)} | parámetros: {n_params:,}")
for lbl in model.configs:
    A = model.transfer_matrices(lbl)
    print(f"  {lbl:10s} P {model.projections[lbl].shape}  "
          f"Q {model.out_maps[lbl].shape}  A_s→d {next(iter(A.values())).shape}")

## 3 · Entrenamiento

In [ ]:
# ---- 3 · Entrenamiento -------------------------------------------------
# Adam lr 1e-3, pérdida MSE estandarizada Z-score por lote + término de
# superficie. Early stopping y reduce-LR según YAML. Reutiliza
# runs/<variante>/best.weights.h5 salvo FORCE=True.
model, history = train_variant(cfg, ds, force=FORCE)
run_dir = Path(cfg.training.run_dir)
print("run_dir:", run_dir)

## 4 · Curvas

In [ ]:
# ---- 4 · Curvas de aprendizaje -----------------------------------------
plot_training(run_dir / "history.csv")
plt.show()

## 5 · Evaluación

In [ ]:
# ---- 5 · Evaluación en test --------------------------------------------
# RMSE/ve por ruta intra-configuración frente a la línea base analítica
# T_d @ pinv(T_s) del propio montaje (columnas *_ana).
metrics_df, summary = evaluate_multiconfig(cfg, ds, model, data)
print("=== RESUMEN (RMSE µV): modelo vs línea base analítica ===")
print(summary.round(3).to_string(index=False))
print("\n=== DETALLE POR CONFIGURACIÓN ===")
print(metrics_df.round(6).to_string(index=False))

## 6 · Superficie

In [ ]:
# ---- 6 · Campo de superficie (heatmap) ---------------------------------
# Métricas del patrón espacial en la malla compartida: rmse_field/r_field/
# ve_field por ruta y configuración.
surface_df, surf_summary = evaluate_multiconfig_surface(model, data)
print("=== CAMPO DE SUPERFICIE (TEST) — resumen ===")
print(surf_summary.round(3).to_string(index=False))
print("\n=== DETALLE POR RUTA ===")
print(surface_df.round(6).to_string(index=False))

## 7 · Figuras

In [ ]:
# ---- 7 · Figuras -------------------------------------------------------
plot_multiconfig_heatmap(metrics_df,
                         title=f"RMSE real multi-config (µV, log10) — {cfg.model.variant}")
plt.show()

plot_multiconfig_bars(metrics_df,
                      title=f"RMSE/ve vs análisis — {cfg.model.variant}")
plt.show()

plot_multiconfig_surface(surface_df,
                         title=f"Campo de superficie (heatmap) — {cfg.model.variant}")
plt.show()

for label, fig in plot_multiconfig_scalps(cfg, model, data=data).items():
    print(f"--- {label} ---")
    plt.show()

## 8 · Guardar

In [ ]:
# ---- 8 · Guardar artefactos en Drive -----------------------------------
# Copia métricas y pesos al run_dir (enlazado a Drive en Colab).
out_dir = run_dir
metrics_df.to_csv(out_dir / "metrics_test.csv", index=False)
surface_df.to_csv(out_dir / "metrics_surface_test.csv", index=False)
summary.to_csv(out_dir / "summary_test.csv", index=False)
model.save_weights(out_dir / "best.weights.h5")
print("Artefactos guardados en:", out_dir)

## Interpretación

* Diagonal (s→s) y cruzada (s→d) intra-configuración: el modelo debe superar
  con claridad a la línea base analítica `*_ana` (la pseudo-inversa rígida
  falla al mezclar subespacios observables distintos).
* `rmse_field`/`ve_field`: calidad del heatmap en la malla; la consistencia
  entre configuraciones (B4 en v2) se evalúa comparando campos de configs
  distintas sobre la misma malla.
* Artefactos en `runs/{variante}/` (Drive en Colab): history.csv,
  metrics_*_test.csv y best.weights.h5.

Documentación: `docs/guia_conceptual.md`, `docs/results_comparison.md`.